### Troisieme methode : utilisation d'un modele de transformer pre-entrainer

In [ ]:
from src.preprocessing import preprocessing, split_data

In [ ]:
df = preprocessing("pcm")
X_train, y_train, X_val,y_val,X_test, y_test = split_data(df)

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict
from transformers import AutoModelForSequenceClassification
import torch
import evaluate

# num_labels=3 pour Positif, Négatif, Neutre
model_name = "castorini/afriberta_large"
device = "cuda" if torch.cuda.is_available() else "cpu"


model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)


# 1. Convertir vos structures de données (listes, séries, etc.) en objets Dataset
train_data = Dataset.from_dict({"text": X_train, "label": y_train})
val_data = Dataset.from_dict({"text": X_val, "label": y_val})
test_data = Dataset.from_dict({"text": X_test, "label": y_test})


# 2. Les regrouper dans un DatasetDict
dataset = DatasetDict({
    "train": train_data,
    "validation": val_data,
    "test"      : test_data
})

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


# 3. Tokeniser les deux splits d'un seul coup
tokenized_datasets = dataset.map(tokenize_function, batched=True)


In [ ]:
# 3. Définir la métrique (ex: Accuracy et F1)
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments

early_stoping = EarlyStoppingCallback(2)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model= "loss"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics = compute_metrics,
    callbacks = [early_stoping]
)

trainer.train()


In [ ]:
results = trainer.evaluate(tokenized_datasets["test"])
print(results)